# M1:Environment and Tooling Milestone

## Docker Compose Usage Guide

### Build & start container (detached)

``` bash
cd eng-ai-agents/project
docker compose up --build -d
```

### See running containers

``` bash
docker ps
```

### Enter the dev container shell

``` bash
docker compose exec graphrag-dev bash
# you will be inside /workspace/project as devuser
source /workspace/.venv/bin/activate

# run development tasks, tests, start servers
python -m your_app_entry_point
# or
uvicorn app:app --host 0.0.0.0 --port 8000
```

### Stop & remove

``` bash
docker compose down
```


# M2: Ingestion Milestone

##### I am using a simple file structure to store the fetched data from the URLs, chunking them and storing the embeddings of the chunks in their respective sub-directories under ```"project/data"```

In [7]:
import requests, os, json, math, hashlib, textwrap
from pathlib import Path
from bs4 import BeautifulSoup


In [ ]:
# # Base directories 
BASE_DIR = Path('./data') 
RAW_DIR = BASE_DIR / 'raw' / 'website'
CLEAN_DIR = BASE_DIR / 'cleaned' / 'website'
CHUNKS_DIR = BASE_DIR / 'chunks' / 'website'

In [9]:
# # Create directories if they don't exist
# for p in (RAW_DIR, CLEAN_DIR, CHUNKS_DIR):
#     p.mkdir(parents=True, exist_ok=True)

# print('Data directories:')
# print(' RAW_DIR ->', RAW_DIR)
# print(' CLEAN_DIR ->', CLEAN_DIR)
# print(' CHUNKS_DIR ->', CHUNKS_DIR)

In [ ]:
# # Wipe and recreate project data directories (safe)
# # This cell will remove directories under BASE_DIR (or './data' if BASE_DIR not defined)
# # Requires interactive confirmation: type YES when prompted.
# from pathlib import Path
# import shutil

# # resolve base directory used by the notebook if present
# try:
#     base_dir = BASE_DIR
# except NameError:
#     base_dir = Path('./data')

# print('Resolved base directory for wipe:', base_dir)

# def wipe_data_dirs(base_dir: Path):
#     base = Path(base_dir)
#     targets = ['raw', 'cleaned', 'chunks', 'embeddings']
#     for d in targets:
#         p = base / d
#         if p.exists():
#             print(f"Removing {p} ...")
#             shutil.rmtree(p)
#         else:
#             print(f"{p} not found, skipping.")
#     # recreate structure (raw/cleaned/chunks have 'website' subfolder)
#     for d in targets:
#         p = base / d
#         if d in ('raw', 'cleaned', 'chunks'):
#             (p / 'website').mkdir(parents=True, exist_ok=True)
#             print(f"Recreated {(p / 'website')}")
#         else:
#             p.mkdir(parents=True, exist_ok=True)
#             print(f"Recreated {p}")

# # Safety: require explicit confirmation
# confirm = False
# # non-interactive override: set confirm_wipe = True in a previous cell to skip prompt
# confirm = bool(globals().get('confirm_wipe', False))

# if not confirm:
#     ans = input(f"Type YES to permanently remove and recreate data directories under {base_dir}: ")
#     confirm = ans.strip() == 'YES'

# if confirm:
#     wipe_data_dirs(base_dir)
# else:
#     print('Aborted. To run non-interactively set confirm_wipe = True and re-run this cell.')


In [11]:
URLS = [
    "https://pantelis.github.io/aiml-common/projects/nlp/ai-tutor/index.html",
    "https://pantelis.github.io/aiml-common/lectures/planning/task-planning/pddl/logistics/",
    "https://pantelis.github.io/aiml-common/lectures/logical-reasoning/automated-reasoning/",
    "https://pantelis.github.io/aiml-common/lectures/VLM/clip/index.html",
    "https://pantelis.github.io/aiml-common/lectures/nlp/transformers/multihead-self-attention.html",
    "https://pantelis.github.io/aiml-common/lectures/nlp/transformers/singlehead-self-attention.html",
    "https://pantelis.github.io/aiml-common/lectures/vae/vae-architecture/"
]

def url_to_id(url: str) -> str:
    h = hashlib.sha1(url.encode()).hexdigest()
    return f"{h}"

print(f"{len(URLS)} URLs provided.")

7 URLs provided.


In [12]:
def fetch_and_save(url: str):
    url_id = url_to_id(url)
    raw_path = RAW_DIR / f"{url_id}.html"
    meta_path = RAW_DIR / f"{url_id}.meta.json"
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        raw_text = resp.text
        raw_path.write_text(raw_text, encoding='utf-8')
        meta = {'url': url, 'status': 'ok', 'status_code': resp.status_code}
        meta_path.write_text(json.dumps(meta, indent=2), encoding='utf-8')
        print(f"Fetched: {url} -> {raw_path.name}")
    except Exception as e:
        meta = {'url': url, 'status': 'error', 'error': str(e)}
        meta_path.write_text(json.dumps(meta, indent=2), encoding='utf-8')
        print(f"Failed to fetch {url}: {e}")

for u in URLS:
    fetch_and_save(u)

Fetched: https://pantelis.github.io/aiml-common/projects/nlp/ai-tutor/index.html -> 33ad566233b66d807b75185f44061735e6d36ec8.html
Fetched: https://pantelis.github.io/aiml-common/lectures/planning/task-planning/pddl/logistics/ -> 66cf67d69038467728100794d05413972aa31260.html
Fetched: https://pantelis.github.io/aiml-common/lectures/logical-reasoning/automated-reasoning/ -> 018589aa1a05b3a4bd8bb47fc17089e7f179f12e.html
Fetched: https://pantelis.github.io/aiml-common/lectures/VLM/clip/index.html -> 8abcc29109d5f6a297c3dacb0b120217e1181246.html
Fetched: https://pantelis.github.io/aiml-common/lectures/nlp/transformers/multihead-self-attention.html -> f677097361114a2aa86d0c9f10120e3e2f00a403.html
Fetched: https://pantelis.github.io/aiml-common/lectures/nlp/transformers/singlehead-self-attention.html -> 1e59f36dde4787fa11544ffd399ead06d0bc5638.html
Fetched: https://pantelis.github.io/aiml-common/lectures/vae/vae-architecture/ -> b9b77d21f6d17163fcaf43505e2dde2416ad1102.html


In [13]:
def clean_html_file(url_id: str):
    raw_path = RAW_DIR / f"{url_id}.html"
    out_path = CLEAN_DIR / f"{url_id}.txt"
    if not raw_path.exists():
        print('raw file not found for', url_id)
        return None
    html = raw_path.read_text(encoding='utf-8')
    soup = BeautifulSoup(html, 'html.parser')

    for s in soup(['script','style','noscript']):
        s.extract()
    text = soup.get_text(separator=' ', strip=True)

    text = ' '.join(text.split())
    out_path.write_text(text, encoding='utf-8')
    print('Cleaned ->', out_path.name)
    return out_path

for meta in RAW_DIR.glob('*.meta.json'):
    m = json.loads(meta.read_text(encoding='utf-8'))
    url = m.get('url')
    uid = url_to_id(url)
    clean_html_file(uid)

Cleaned -> 33ad566233b66d807b75185f44061735e6d36ec8.txt
Cleaned -> b9b77d21f6d17163fcaf43505e2dde2416ad1102.txt
Cleaned -> 018589aa1a05b3a4bd8bb47fc17089e7f179f12e.txt
Cleaned -> 66cf67d69038467728100794d05413972aa31260.txt
Cleaned -> 1e59f36dde4787fa11544ffd399ead06d0bc5638.txt
Cleaned -> 8abcc29109d5f6a297c3dacb0b120217e1181246.txt
Cleaned -> f677097361114a2aa86d0c9f10120e3e2f00a403.txt


In [14]:
def chunk_text(text: str, chunk_size_words=200, overlap_words=50):
    words = text.split()
    if not words:
        return []
    chunks = []
    step = chunk_size_words - overlap_words
    for i in range(0, len(words), step):
        chunk_words = words[i:i+chunk_size_words]
        if not chunk_words:
            break
        chunk = ' '.join(chunk_words)
        chunks.append(chunk)
        if i + chunk_size_words >= len(words):
            break
    return chunks

def chunk_cleaned_file(url_id: str, chunk_size_words=200, overlap_words=50):
    in_path = CLEAN_DIR / f"{url_id}.txt"
    if not in_path.exists():
        print('cleaned file missing for', url_id)
        return 0
    text = in_path.read_text(encoding='utf-8')
    chunks = chunk_text(text, chunk_size_words, overlap_words)
    saved = 0
    for idx, ch in enumerate(chunks):
        chunk_meta = {
            'url_id': url_id,
            'chunk_index': idx,
            'chunk_text': ch,
        }
        chunk_path = CHUNKS_DIR / f"{url_id}_chunk_{idx}.json"
        chunk_path.write_text(json.dumps(chunk_meta, ensure_ascii=False), encoding='utf-8')
        saved += 1
    print(f'Saved {saved} chunks for', url_id)
    return saved

for txt in CLEAN_DIR.glob('*.txt'):
    uid = txt.stem
    chunk_cleaned_file(uid, chunk_size_words=100, overlap_words=20)

Saved 18 chunks for 66cf67d69038467728100794d05413972aa31260
Saved 9 chunks for f677097361114a2aa86d0c9f10120e3e2f00a403
Saved 29 chunks for 1e59f36dde4787fa11544ffd399ead06d0bc5638
Saved 39 chunks for 8abcc29109d5f6a297c3dacb0b120217e1181246
Saved 9 chunks for 018589aa1a05b3a4bd8bb47fc17089e7f179f12e
Saved 16 chunks for b9b77d21f6d17163fcaf43505e2dde2416ad1102
Saved 17 chunks for 33ad566233b66d807b75185f44061735e6d36ec8


In [15]:
import json
import requests
from pathlib import Path

CHUNKS_DIR = Path("data/chunks/website")
EMBEDS_DIR = Path("data/embeddings")
EMBEDS_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = "http://host.docker.internal:11434/api/embeddings"
MODEL = "all-minilm:l6-v2"

def embed_text_ollama(text: str):
    payload = {"model": MODEL, "prompt": text}
    r = requests.post(OLLAMA_URL, json=payload, timeout=30)
    r.raise_for_status()
    return r.json()["embedding"]

def embed_all_chunks():
    for chunk_file in CHUNKS_DIR.glob("*.json"):
        out_path = EMBEDS_DIR / chunk_file.name
        print(out_path)
        if out_path.exists():
            continue  
        data = json.loads(chunk_file.read_text(encoding="utf-8"))
        text = data["chunk_text"]
        print(len(text.split()), "words in chunk")
        try:
            emb = embed_text_ollama(text)
        except Exception as e:
            print(f"Failed to embed {chunk_file.name}: {e}")
            continue
        data["embedding"] = emb
        out_path.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
        print(f"Embedded {chunk_file.name}")

embed_all_chunks()


data/embeddings/8abcc29109d5f6a297c3dacb0b120217e1181246_chunk_27.json
data/embeddings/66cf67d69038467728100794d05413972aa31260_chunk_13.json
data/embeddings/f677097361114a2aa86d0c9f10120e3e2f00a403_chunk_1.json
data/embeddings/1e59f36dde4787fa11544ffd399ead06d0bc5638_chunk_7.json
data/embeddings/b9b77d21f6d17163fcaf43505e2dde2416ad1102_chunk_9.json
data/embeddings/1e59f36dde4787fa11544ffd399ead06d0bc5638_chunk_11.json
data/embeddings/8abcc29109d5f6a297c3dacb0b120217e1181246_chunk_31.json
data/embeddings/018589aa1a05b3a4bd8bb47fc17089e7f179f12e_chunk_6.json
data/embeddings/33ad566233b66d807b75185f44061735e6d36ec8_chunk_15.json
data/embeddings/b9b77d21f6d17163fcaf43505e2dde2416ad1102_chunk_5.json
data/embeddings/66cf67d69038467728100794d05413972aa31260_chunk_0.json
data/embeddings/8abcc29109d5f6a297c3dacb0b120217e1181246_chunk_7.json
data/embeddings/33ad566233b66d807b75185f44061735e6d36ec8_chunk_1.json
data/embeddings/8abcc29109d5f6a297c3dacb0b120217e1181246_chunk_11.json
data/embedding

In [6]:
import json, numpy as np
from pathlib import Path

def load_embedding_file(p: Path):
    data = {}
    try:
        txt = p.read_text(encoding='utf-8')
        data = json.loads(txt)
        return data
    except Exception:
        pass
    try:
        arr = np.load(p)
        return {"embedding": arr.tolist(), "chunk_text": None, "source": str(p)}
    except Exception:
        pass

    try:
        txt = p.read_text(encoding='utf-8')
        return {"chunk_text": txt, "embedding": None, "source": str(p)}
    except Exception as e:
        print("Could not load", p, e)
        return None

# embed_files = sorted(EMBEDS_DIR.glob("*"))
# loaded = []
# for f in embed_files:
#     d = load_embedding_file(f)
#     if d is None:
#         continue
#     if "chunk_text" not in d and "text" in d:
#         d["chunk_text"] = d.pop("text")
#     if "embedding" not in d:
#         d["embedding"] = None
#     d["_file"] = str(f)
#     d["_id"] = Path(f).stem
#     loaded.append(d)

# print(f"Loaded {len(loaded)} embedding files (sample keys):", [x['_id'] for x in loaded[:5]])

# Path("data").mkdir(parents=True, exist_ok=True)
# Path("data/loaded_embeddings.json").write_text(json.dumps(loaded[:50], indent=2, ensure_ascii=False), encoding='utf-8')

# M3: GraphRAG Construction Milestone

In [ ]:
import aiohttp
import json
from typing import List, Dict, Any, Union
import asyncio

OLLAMA_HOST = "http://host.docker.internal:11434"
EMBED_MODEL = "all-minilm:l6-v2"
CHAT_MODEL = "qwen2"
EMBED_URL = f"{OLLAMA_HOST}/api/embeddings"
CHAT_URL = f"{OLLAMA_HOST}/api/chat"



_session = None

async def get_session():
    global _session
    if _session is None:
        _session = aiohttp.ClientSession()
    return _session

### fix
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
EMBED_MAX_TOKENS = 180


def truncate_to_256_tokens(text: str) -> str:
    tokens = enc.encode(text)
    if len(tokens) <= EMBED_MAX_TOKENS:
        return text
    return enc.decode(tokens[:EMBED_MAX_TOKENS])



async def embed_text_ollama(text_or_texts: Union[str, List[str]]):
    session = await get_session()

    if isinstance(text_or_texts, str):
        text = truncate_to_256_tokens(text_or_texts)
        payload = {"model": EMBED_MODEL, "prompt": text}

        async with session.post(EMBED_URL, json=payload, timeout=60) as resp:
            resp.raise_for_status()
            return (await resp.json())["embedding"]

    if isinstance(text_or_texts, list):
        async def embed_one(text):
            text = truncate_to_256_tokens(text)
            payload = {"model": EMBED_MODEL, "prompt": text}
            async with session.post(EMBED_URL, json=payload, timeout=60) as resp:
                resp.raise_for_status()
                return (await resp.json())["embedding"]

        tasks = [embed_one(t) for t in text_or_texts]
        return await asyncio.gather(*tasks)


embed_text_ollama.embedding_dim = 384


In [ ]:
async def ollama_chat(prompt: str, system_prompt: str = None, history_messages: List[Dict[str,str]] = [], **kwargs) -> str:
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": CHAT_MODEL,
        "messages": messages,
        "stream": False,
        "temperature":0.9,
        "num_predict": 2048
    }

    async with aiohttp.ClientSession() as session:
        async with session.post(CHAT_URL, json=payload, timeout=2000) as response:
            response.raise_for_status()
            data = await response.json()
            return data["message"]["content"]


from nano_graphrag import GraphRAG
from pathlib import Path

rag = GraphRAG(
    working_dir="./graphrag_website_new",
    cheap_model_func=ollama_chat,  
    best_model_func=ollama_chat,  
    embedding_func=embed_text_ollama,
)

print("Custom Ollama LLM + embeddings registered successfully")

CHUNKS_DIR = Path("data/chunks/website")

async def run_insert(batch_size=8):
    files = list(CHUNKS_DIR.glob("*.json"))

    for i in range(0, len(files), batch_size):
        batch_files = files[i:i+batch_size]
        texts = []

        for f in batch_files:
            chunk = json.loads(f.read_text())
            texts.append(chunk["chunk_text"])

        await rag.ainsert(texts)



await run_insert()


# M4: Query and Generation Milestone

##### Approach

When a user submits a query, the system first runs **embedding retrieval** to collect a small candidate set — this keeps the search efficient but broad enough to avoid missing relevant context. From there, I expand through the graph through traversal: hyperparameters like `max_candidates`, `traversal_depth`, `max_siblings`, and `max_children` act as guardrails so the system only explores meaningful semantic neighborhoods rather than the entire graph. This design considers **entity relationships** during traversal. The depth parameter ensures that it captures **multi-hop reasoning** without drifting too far, while sibling limits prevent the explosion of loosely related nodes. Once this semantically rich subgraph is assembled, I use a **structured context builder** to summarize nodes and feed them to the LLM for final reasoning.


In [ ]:
import asyncio, json, textwrap, re, ast
from typing import List, Dict, Any
from pathlib import Path
import networkx as nx

MAX_CANDIDATES = 6
MAX_PREREQ_DEPTH = 4
MAX_SIBLINGS = 6
MAX_RESOURCES = 3
MAX_EXAMPLES = 3
MAX_PROMPT_CONCEPTS = 5

def _strip_quotes(s: str):
    if not isinstance(s, str): return s
    return s.strip().strip('"').strip("'")

def _norm(s: str) -> str:
    return re.sub(r'\s+', ' ', (s or "").strip().lower())

def _safe_trunc(s: str, n=300):
    if not s: return ""
    return s if len(s) <= n else s[:n].rsplit(' ',1)[0] + "…"

def strip_outer_quotes(s: str) -> str:
    if not isinstance(s, str):
        return s
    s = s.strip()
    if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
        return s[1:-1].strip()
    return s

def clean_concept_for_prompt(node_meta: dict, node_id: str):
    title = node_meta.get("title") or node_meta.get("entity_name") or node_meta.get("name") or node_id
    title = strip_outer_quotes(title)
    defs = node_meta.get("definitions") or node_meta.get("description") or node_meta.get("text") or ""
    return title, _safe_trunc(strip_outer_quotes(defs), 300)

def _ensure_nx_graph(rag):
    storage = getattr(rag, "chunk_entity_relation_graph", None)
    if storage is not None:
        for cand in ("nx_graph", "G", "graph", "networkx_graph"):
            g = getattr(storage, cand, None)
            if g is not None:
                return nx.DiGraph(g)
        for method in ("get_graph", "to_networkx"):
            if hasattr(storage, method):
                try:
                    g = getattr(storage, method)()
                    return nx.DiGraph(g)
                except Exception:
                    pass
    wk = getattr(rag, "working_dir", None) or "./graphrag_website"
    for p in (Path(wk) / "graph_chunk_entity_relation.graphml",
              Path(wk) / "graph_chunk_entity_relation.graphml.gz",
              Path("./graphrag_website/graph_chunk_entity_relation.graphml")):
        if p.exists():
            try:
                g = nx.read_graphml(str(p))
                return nx.DiGraph(g)
            except Exception:
                pass
    raise RuntimeError("Could not load networkx graph from rag.working_dir or wrapper")

_KV_CHUNKS_CACHE: Dict[str, Dict[str, Any]] = None

def load_kv_chunks_map(rag) -> Dict[str, Dict[str,Any]]:
    """
    Return mapping chunk_id -> chunk record (dict). The input file typically is kv_store_text_chunks.json
    """
    global _KV_CHUNKS_CACHE
    if _KV_CHUNKS_CACHE is not None:
        return _KV_CHUNKS_CACHE
    wk = getattr(rag, "working_dir", None) or "./graphrag_website"
    cand_paths = [
        Path(wk) / "kv_store_text_chunks.json",
        Path("./graphrag_website/kv_store_text_chunks.json"),
        Path("./kv_store_text_chunks.json"),
    ]
    for p in cand_paths:
        if p.exists():
            try:
                data = json.loads(p.read_text(encoding="utf-8"))

                if isinstance(data, list):

                    _KV_CHUNKS_CACHE = {d.get("chunk_id") or d.get("id") or d.get("_id"): d for d in data if isinstance(d, dict)}
                elif isinstance(data, dict):

                    _KV_CHUNKS_CACHE = data
                else:
                    _KV_CHUNKS_CACHE = {}
                return _KV_CHUNKS_CACHE
            except Exception as e:
                print("load_kv_chunks_map: failed to parse", p, e)
    _KV_CHUNKS_CACHE = {}
    return _KV_CHUNKS_CACHE

def chunk_snippet_from_source_id(source_id: str, kv_chunks: Dict[str,Dict[str,Any]], chars=400) -> str:
    """
    source_id may be "chunk-abc<SEP>chunk-def" or "chunk-abc|chunk-def". Return first matching snippet.
    """
    if not source_id:
        return ""
    parts = re.split(r'\<SEP\>|\||,|\s+', str(source_id))
    snippets = []
    for p in parts:
        if not p: continue
        rec = kv_chunks.get(p)
        if not rec:

            continue

        text = rec.get("chunk_text") or rec.get("text") or rec.get("content") or rec.get("body") or ""
        if text:
            snippets.append(_safe_trunc(text.strip(), chars))

    return "\n\n".join(snippets[:2]) if snippets else ""


def resolve_candidate_to_node_id(rag, candidate: Dict[str,Any]):
    try:
        G = _ensure_nx_graph(rag)
    except Exception as e:
        print("[resolve] cannot load graph:", e)
        return None
    nodes = list(G.nodes())

    title_map = {}
    for n in nodes:
        md = dict(G.nodes[n])
        for key in ("title", "entity_name", "name", "label", "text"):
            val = md.get(key)
            if isinstance(val, str) and val.strip():
                title_map[_norm(_strip_quotes(val))] = n
        title_map[_norm(_strip_quotes(str(n)))] = n

        for k in ("vector_id", "vdb_id", "ent_id", "entity_id", "_id", "id"):
            if k in md and md[k] is not None:
                title_map[_norm(str(md[k]))] = n

    raw = candidate.get("raw", {}) or {}
    candidate_node_id = candidate.get("node_id")

    if candidate_node_id in nodes:
        return candidate_node_id

    for key in ("entity_node_id", "node_id", "node", "graph_node_id", "map_to"):
        if key in raw and raw[key] in nodes:
            return raw[key]

    meta = raw.get("meta") or raw.get("metadata") or raw
    if isinstance(meta, dict):
        for key in ("entity_name", "title", "name", "label", "id"):
            val = meta.get(key)
            if isinstance(val, str) and _norm(_strip_quotes(val)) in title_map:
                return title_map[_norm(_strip_quotes(val))]

    if isinstance(candidate_node_id, str):
        cand_norm = _norm(_strip_quotes(candidate_node_id))
        if cand_norm in title_map:
            return title_map[cand_norm]
        for k,v in title_map.items():
            if cand_norm and (cand_norm in k or k in cand_norm):
                return v

    for fk in ("source", "file", "source_url", "chunk_file", "_file"):
        val = meta.get(fk) if isinstance(meta, dict) else None
        if val:
            val_norm = _norm(str(val))
            for n in nodes:
                md = dict(G.nodes[n])
                for nk in ("source", "file", "source_url", "chunk_file", "_file"):
                    if _norm(str(md.get(nk, ""))) == val_norm:
                        return n

    search_text = ""
    for k in ("text","chunk_text","content","title","description"):
        if k in raw and isinstance(raw[k], str):
            search_text += " " + raw[k]
    search_text = _norm(search_text)
    if search_text:
        for k,v in title_map.items():
            if k and k in search_text:
                return v

    return None

def build_subgraph_for_node(rag, node_id: str) -> Dict[str,Any]:
    G = _ensure_nx_graph(rag)
    if node_id not in G:
        print(f"[debug] node_id {node_id} not found in graph")
        return {}

    nodes = {}
    edges = []

    nodes[node_id] = dict(G.nodes[node_id])
    central_meta = nodes[node_id]

    prereqs = []
    seen = set()
    def walk_prereq(n, depth):
        if depth <= 0:
            return
        for u, v, d in G.in_edges(n, data=True):
            et = (d.get("type") or d.get("relation") or d.get("label") or "").lower()
            if "prereq" in et or "requires" in et or "precedes" in et:
                if u not in seen:
                    seen.add(u)
                    nodes[u] = dict(G.nodes[u])
                    edges.append({"source": u, "target": n, "type": et or "prereq"})
                    prereqs.append(u)
                    walk_prereq(u, depth-1)
    walk_prereq(node_id, MAX_PREREQ_DEPTH)
    prereqs = list(reversed(prereqs))

    siblings = []
    for u, v, d in list(G.edges(node_id, data=True)):
        et = (d.get("type") or d.get("relation") or d.get("label") or "").lower()
        if any(k in et for k in ("near", "similar", "related", "sibling", "contrast", "part_of", "is_a")):
            if v not in nodes: nodes[v] = dict(G.nodes[v])
            edges.append({"source": node_id, "target": v, "type": et or "related"})
            siblings.append(v)
    if len(siblings) < 1:
        for nbr in list(G.neighbors(node_id))[:MAX_SIBLINGS]:
            if nbr not in nodes: nodes[nbr] = dict(G.nodes[nbr])
            edges.append({"source": node_id, "target": nbr, "type": "neighbor"})
            siblings.append(nbr)
    siblings = siblings[:MAX_SIBLINGS]

    resources = []
    for u, v, d in G.in_edges(node_id, data=True):
        et = (d.get("type") or d.get("relation") or d.get("label") or "").lower()
        if any(k in et for k in ("explain", "reference", "source", "cite", "mentions")) or str(G.nodes[u].get("type","")).lower().startswith("resource"):
            nodes[u] = dict(G.nodes[u])
            edges.append({"source": u, "target": node_id, "type": et or "explains"})
            resources.append(u)
    resources = resources[:MAX_RESOURCES]

    examples = []
    for u, v, d in G.in_edges(node_id, data=True):
        et = (d.get("type") or d.get("relation") or d.get("label") or "").lower()
        if "example" in et or str(G.nodes[u].get("type","")).lower() == "example":
            nodes[u] = dict(G.nodes[u])
            edges.append({"source": u, "target": node_id, "type": et or "exemplifies"})
            examples.append(u)
    examples = examples[:MAX_EXAMPLES]

    sub = {
        "concept_id": node_id,
        "concept_meta": central_meta,
        "prereqs": prereqs,
        "siblings": siblings,
        "resources": resources,
        "examples": examples,
        "nodes": nodes,
        "edges": edges
    }

    kv = load_kv_chunks_map(rag)
    if not sub["resources"]:
        possible_source = central_meta.get("source_id") or central_meta.get("source") or central_meta.get("_file") or central_meta.get("file")
        snippet = chunk_snippet_from_source_id(possible_source or "", kv, chars=400)
        if snippet:
            fake_id = f"chunk::{(possible_source or 'unknown')}"
            sub["nodes"][fake_id] = {
                "title": f"source snippet for {strip_outer_quotes(node_id)}",
                "text": snippet,
                "source": possible_source
            }
            sub["resources"].append(fake_id)
            sub["edges"].append({"source": fake_id, "target": node_id, "type": "explains (chunk-snippet)"})

    return sub

def parse_llm_json_response(resp_text: str, max_unwrap=2):
    if not isinstance(resp_text, str):
        return None
    attempt = resp_text
    for _ in range(max_unwrap + 1):
        try:
            parsed = json.loads(attempt)
        except Exception:
            try:
                parsed = ast.literal_eval(attempt)
            except Exception:
                parsed = None
        if isinstance(parsed, dict):

            for k, v in list(parsed.items()):
                if isinstance(v, str) and v.strip().startswith(('{','[')) and v.strip().endswith(('}',']')):
                    try:
                        parsed[k] = json.loads(v)
                    except Exception:
                        pass
            return parsed
        if isinstance(parsed, str):
            attempt = parsed
            continue
        break
    return None

async def semantic_candidates(rag, query: str, top_k: int = MAX_CANDIDATES) -> List[Dict[str,Any]]:
    qs = query.strip()

    try:
        if hasattr(rag, "entities_vdb") and hasattr(rag.entities_vdb, "query"):
            res = await rag.entities_vdb.query(qs, top_k=top_k)
            if res is None:
                raise RuntimeError("entities_vdb.query returned None")
            candidates = []
            for r in res:
                node_id = r.get("id") or r.get("_id") or r.get("entity_id") or r.get("entity_name") or r.get("meta",{}).get("entity_name")
                score = r.get("score") if "score" in r else r.get("distance") or r.get("sim") or None
                candidates.append({"node_id": node_id, "score": score, "raw": r})
            candidates = [c for c in candidates if c["node_id"]]
            try:
                candidates = sorted(candidates, key=lambda x: float(x["score"]) if x.get("score") is not None else 0.0, reverse=True)
            except Exception:
                pass
            if candidates:
                print(f"[debug] semantic_candidates (via entities_vdb.query) -> {[c['node_id'] for c in candidates]}")
                return candidates[:top_k]
    except Exception as e:
        print("[debug] entities_vdb.query failed:", e)

    try:
        emb_func = getattr(rag, "embedding_func", None) or getattr(rag, "embedding", None)
        if emb_func is None:
            emb_func = getattr(rag.entities_vdb, "embedding_func", None)
        if emb_func is None:
            raise RuntimeError("No embedding function found on rag")
        if asyncio.iscoroutinefunction(emb_func):
            emb = await emb_func(qs)
        else:
            loop = asyncio.get_event_loop()
            emb = await loop.run_in_executor(None, emb_func, qs)
        if isinstance(emb, list) and len(emb) and isinstance(emb[0], list):
            emb_vec = emb[0]
        else:
            emb_vec = emb
        client = getattr(rag.entities_vdb, "_client", None)
        if client is None:
            raise RuntimeError("No low-level _client on entities_vdb")
        loop = asyncio.get_event_loop()
        results = await loop.run_in_executor(None, client.query, emb_vec, top_k, getattr(rag.entities_vdb, "cosine_better_than_threshold", None))
        cand = []
        for r in results:
            nid = r.get("id") or r.get("_id") or r.get("meta", {}).get("entity_name") or r.get("entity_name")
            score = r.get("score") or r.get("distance") or None
            cand.append({"node_id": nid, "score": score, "raw": r})
        if cand:
            print(f"[debug] semantic_candidates (via low-level client) -> {[c['node_id'] for c in cand]}")
            return cand[:top_k]
    except Exception as e:
        print("[debug] fallback embedding->client failed:", e)

    try:
        G = _ensure_nx_graph(rag)
        node_meta = {n: dict(G.nodes[n]) for n in G.nodes()}
        sample = []
        for nid, md in list(node_meta.items())[:200]:
            title = md.get("title") or md.get("entity_name") or nid
            defs = md.get("definitions") or md.get("definition") or md.get("description") or ""
            sample.append(f"{title} ||| id={nid} ||| def={_safe_trunc(defs,120)}")
        nodes_context = "\n".join(sample)
        prompt = textwrap.dedent(f"""
            Choose up to {top_k} node ids from the following list that best match the student's question.
            Return a JSON array of ids only.

            KG_NODES:
            {nodes_context}

            Student question:
            {query}
        """)
        best_fn = getattr(rag, "best_model_func", None) or getattr(rag, "cheap_model_func", None)
        if best_fn is None:
            raise RuntimeError("No LLM function is available on rag")
        hk = getattr(rag, "llm_response_cache", None)
        resp = await best_fn(prompt, system_prompt=None, history_messages=[], hashing_kv=hk)
        cand_ids = []
        try:
            cand_ids = json.loads(resp)
            if not isinstance(cand_ids, list):
                cand_ids = []
        except Exception:
            cand_ids = re.findall(r'["\']([A-Za-z0-9_\-\. ]+)["\']', resp)
        out = [{"node_id": cid, "score": None, "raw": None} for cid in cand_ids if cid]
        print("[debug] LLM fallback candidates:", [c['node_id'] for c in out])
        return out[:top_k]
    except Exception as e:
        print("[debug] final fallback failed:", e)
        return []

async def generate_answer(rag, query: str, candidate_nodes: List[str], learner_level="introductory"):
    G = _ensure_nx_graph(rag)
    kv_map = load_kv_chunks_map(rag)

    subgraphs = []
    references = []
    for nid in candidate_nodes[:MAX_PROMPT_CONCEPTS]:
        sg = build_subgraph_for_node(rag, nid)
        if not sg:
            continue
        subgraphs.append(sg)

        meta = sg.get("concept_meta", {})
        title = strip_outer_quotes(meta.get("title") or meta.get("entity_name") or nid)
        short_def = _safe_trunc(strip_outer_quotes(meta.get("definitions") or meta.get("description") or meta.get("text") or ""), 300)

        evidence_text = ""
        if sg.get("resources"):
            first_r = sg["resources"][0]
            rmeta = sg["nodes"].get(first_r, {})

            evidence_text = rmeta.get("text") or rmeta.get("chunk_text") or ""
            if not evidence_text:
                src = rmeta.get("source") or rmeta.get("source_id") or rmeta.get("file") or meta.get("source_id")
                if src:
                    evidence_text = chunk_snippet_from_source_id(src, kv_map, chars=350)
            if not evidence_text:
                # as a last fallback, try node's description
                evidence_text = strip_outer_quotes(rmeta.get("description") or rmeta.get("definitions") or "")
        references.append({"concept": title, "evidence": _safe_trunc(evidence_text, 350) if evidence_text else ""})

    concept_sections = []
    for sg in subgraphs:
        nid = sg["concept_id"]
        meta = sg["concept_meta"]
        title = strip_outer_quotes(meta.get("title") or meta.get("entity_name") or nid)
        short_def = _safe_trunc(strip_outer_quotes(meta.get("definitions") or meta.get("description") or meta.get("text") or ""), 250)
        evids = []
        for rnode in sg.get("resources", [])[:2]:
            rmeta = sg["nodes"].get(rnode, {})
            ev = rmeta.get("text") or rmeta.get("chunk_text") or rmeta.get("description") or ""
            if not ev:
                src = rmeta.get("source") or rmeta.get("source_id") or meta.get("source_id")
                if src:
                    ev = chunk_snippet_from_source_id(src, kv_map, chars=300)
            if ev:
                evids.append(_safe_trunc(ev, 300))
        concept_sections.append(f"### {title}\n{short_def}\n\nEvidence:\n" + ("\n\n".join(evids) if evids else "(no snippet available)") + "\n\n")

        prompt = f"""
You are an expert AI tutor. Your job is to answer any technical questions the student asks using 
clear reasoning, with mathematics and code examples whenever they meaningfully improve understanding.

Where appropriate:
- Explain concepts first in simple terms, then add technical depth.
- Provide mathematical expressions for key equations.
- Provide short Python examples when they clarify the mechanism or computation.
- If concepts involve comparisons, mechanisms, steps, or inner workings, give structured paragraphs.

You must base your answers ONLY on the provided concept summaries and exact evidence snippets 
extracted from the knowledge graph. Do NOT invent new facts.

After each question is answered, you MUST:
1. List the knowledge graph nodes used.
2. List the references or evidence snippets used.
4. Provide 1-2 short practice questions.


Student asked:
\"\"\"{query}\"\"\"

Learner level: {learner_level}

Concepts available:
{''.join(concept_sections)}
"""


    best_fn = getattr(rag, "best_model_func", None) or getattr(rag, "cheap_model_func", None)
    if best_fn is None:
        raise RuntimeError("No LLM function available on rag for generation")
    resp = await best_fn(prompt, system_prompt=None, history_messages=[], hashing_kv=getattr(rag, "llm_response_cache", None))

    parsed = parse_llm_json_response(resp)
    text_answer = None
    if isinstance(parsed, dict) and parsed.get("answer"):
        text_answer = parsed["answer"]
    elif isinstance(parsed, dict) and any(isinstance(v, str) for v in parsed.values()):
        joinable = " ".join([v for v in parsed.values() if isinstance(v, str)])
        text_answer = joinable.strip() if joinable.strip() else None
    if not text_answer:
        text_answer = resp.strip()

    return {
        "answer": text_answer,
        "used_concepts": [sg["concept_id"] for sg in subgraphs],
        "references": references,
        "subgraphs": subgraphs
    }

async def query_and_answer(rag, query: str, top_k=MAX_CANDIDATES, learner_level="introductory"):
    cand = await semantic_candidates(rag, query, top_k=top_k)
    if not cand:
        return {"answer": "No semantic candidates found", "used_concepts": [], "references": [], "subgraphs": []}

    resolved = []
    for c in cand:
        try:
            node = resolve_candidate_to_node_id(rag, c)
        except Exception as e:
            print("[warn] resolver error:", c.get("node_id"), e)
            node = None
        if node and node not in resolved:
            resolved.append(node)

    if not resolved:
        candidate_ids = [c["node_id"] for c in cand if c.get("node_id")]
        return await generate_answer(rag, query, candidate_ids, learner_level=learner_level)

    return await generate_answer(rag, query, resolved, learner_level=learner_level)


result = await query_and_answer(rag, "What is CLIP and how it is used in computer vision applications?", top_k=6, learner_level="intermediate")
print(json.dumps(result, indent=2))

[debug] length of text:1 
[debug] semantic_candidates (via entities_vdb.query) -> ['ent-2fc75946c403ba3e09711934542b87db', 'ent-a30c298dc48adf70da00883278fa9f2d', 'ent-2a25a15730ece1e6bd7358965f8f84f7', 'ent-fcc67bf39b9fae158da8286b5e8244d2', 'ent-7b3b00cee984ac03a12b1ab82ecaeb38', 'ent-507f1ba4220d0d18b512b8918ca14c76']
{
  "answer": "### Knowledge Graph Nodes Used:\n\n1. CLIP Architecture\n2. CONTRASTIVE LANGUAGE-IMAGE PRETRAINING (CLIP)\n3. CLIP Paper\n4. Vision Transformer\n\n### References and Evidence Snippets:\n\n* CLIP connects domains of images and language, enhancing the applicability of language-based abilities to imagery.\n* CLIP is a method that pretrains models to understand and match relationships between textual descriptions and visual images.\n\n### Detailed Explanation:\n\nCLIP (Contrastive Language-Image Pretraining) is an innovative architecture developed for bridging the gap between image processing and text understanding in computer vision tasks. It achieves this 

In [ ]:
from IPython.display import Markdown

Markdown(rf"""{result["answer"]}""")


### Knowledge Graph Nodes Used:

1. CLIP Architecture
2. CONTRASTIVE LANGUAGE-IMAGE PRETRAINING (CLIP)
3. CLIP Paper
4. Vision Transformer

### References and Evidence Snippets:

* CLIP connects domains of images and language, enhancing the applicability of language-based abilities to imagery.
* CLIP is a method that pretrains models to understand and match relationships between textual descriptions and visual images.

### Detailed Explanation:

CLIP (Contrastive Language-Image Pretraining) is an innovative architecture developed for bridging the gap between image processing and text understanding in computer vision tasks. It achieves this by training a model on large-scale datasets containing pairs of images accompanied by corresponding descriptive captions or questions. The goal is to enable the model to understand how well its visual predictions align with textual descriptions, thus enhancing multimodal reasoning abilities.

CLIP relies on a Transformer-based architecture for both text and image encoders, allowing it to process language (texts) and visual information (images). Specifically, CLIP can take input in two forms:

1. **Textual Input**: This could be natural language instructions or descriptions about objects.
2. **Visual Input**: Images depicting those described objects.

The core principle of CLIP is based on contrastive learning, which aims to maximize the similarity scores between images and their corresponding text descriptions while minimizing the scores for other unrelated pairs. It does this through a process called contrastive loss computation:

\[ \text{Contrastive Loss} = -\log \left( \frac{\exp(s_{ij})}{\sum_k \exp(s_{ik})} \right) \]

where \( s_{ij} \) represents the similarity score between image \( i \) and text \( j \), and \( k \) iterates over all pairs of images-text descriptions.

To understand how CLIP works, imagine training it with a dataset containing photos of various animals alongside their names written in natural language. The model would learn to assign higher scores when matching the correct name with an image of that animal (e.g., "dog" paired with an actual dog photo) and lower scores for incorrect matches.

### Practice Questions:

1. **Question**: Explain how CLIP uses attention mechanisms within its Transformer architecture.
2. **Question**: Describe a scenario where you would use CLIP in computer vision applications, providing specific text input (e.g., "a red apple") and image description ("an apple with a red skin").